# VQ-VAE v14 Training - Rebalanced Loss Weights

## Overview
v14 rebalances loss weights to allow the model freedom to adjust air/structure boundaries.

## The Problem (v13a/v13b)
- v13a (Gumbel): 52.4% acc, 1.342x volume - Gumbel too soft, argmax still over-predicts
- v13b (STE): 7.3% acc, 1.233x volume - STE improved volume but accuracy collapsed

**Root cause**: 20x asymmetric weight conflicts with volume loss. Model can't adjust boundaries.

## The v14 Solution: Rebalanced Weights

| Parameter | v13b | v14 | Rationale |
|-----------|------|-----|----------|
| Structure->Air weight | 20.0x | **6.0x** | Allow boundary adjustments |
| False air penalty | 10.0x | **5.0x** | Was redundant |
| Hard volume weight | 10.0x | **20.0x** | Make volume the priority |

**Key insight**: v13b proved STE works (1.307x -> 1.233x). We just need to let the model adjust boundaries.

**Trade-off**: Accept lower recall (99.9% -> ~90%) for correct volume ratio.

## 1. Setup - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_BASE = '/content/drive/MyDrive/minecraft_ai'
OUTPUT_DIR = '/content/drive/MyDrive/minecraft_ai/vqvae_v14'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# === Copy data to local storage for faster training ===
print("\nCopying data to local storage (this takes ~2-3 min but speeds up training significantly)...")
!cp -r /content/drive/MyDrive/minecraft_ai/splits /content/splits
!cp -r /content/drive/MyDrive/minecraft_ai/vocabulary /content/vocabulary
!cp -r /content/drive/MyDrive/minecraft_ai/embeddings /content/embeddings
print("Data copied to local storage!")

## 2. Imports

In [ ]:
import json
import random
import time
from pathlib import Path
from typing import Dict, List, Tuple, Set, Optional
from collections import defaultdict

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. RFSQ Quantization Module

In [ ]:
class FSQ(nn.Module):
    """Finite Scalar Quantization - quantizes each dimension to fixed levels."""
    def __init__(self, levels: List[int], eps: float = 1e-3):
        super().__init__()
        self.levels = levels
        self.dim = len(levels)
        self.eps = eps
        self.codebook_size = int(np.prod(levels))
        self.register_buffer('_levels', torch.tensor(levels, dtype=torch.float32))
        basis = []
        acc = 1
        for L in reversed(levels):
            basis.append(acc)
            acc *= L
        self.register_buffer('_basis', torch.tensor(list(reversed(basis)), dtype=torch.long))
        half_levels = [(L - 1) / 2 for L in levels]
        self.register_buffer('_half_levels', torch.tensor(half_levels, dtype=torch.float32))

    def forward(self, z: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        z_bounded = torch.tanh(z)
        z_q = self._quantize(z_bounded)
        z_q = z_bounded + (z_q - z_bounded).detach()  # Straight-through
        indices = self._to_indices(z_q)
        return z_q, indices

    def _quantize(self, z: torch.Tensor) -> torch.Tensor:
        z_q_list = []
        for i in range(self.dim):
            half_L = self._half_levels[i]
            z_i = z[..., i] * half_L
            z_i = torch.round(z_i).clamp(-half_L, half_L) / half_L
            z_q_list.append(z_i)
        return torch.stack(z_q_list, dim=-1)

    def _to_indices(self, z_q: torch.Tensor) -> torch.Tensor:
        indices = torch.zeros(z_q.shape[:-1], dtype=torch.long, device=z_q.device)
        for i in range(self.dim):
            L = self._levels[i].long()
            half_L = self._half_levels[i]
            level_idx = ((z_q[..., i] * half_L) + half_L).round().long().clamp(0, L - 1)
            indices = indices + level_idx * self._basis[i]
        return indices

    def get_codebook_usage(self, indices: torch.Tensor) -> Tuple[float, float]:
        flat = indices.flatten()
        counts = torch.bincount(flat, minlength=self.codebook_size).float()
        usage = (counts > 0).float().mean().item()
        probs = counts / counts.sum()
        probs = probs[probs > 0]
        entropy = -(probs * torch.log(probs)).sum()
        perplexity = torch.exp(entropy).item()
        return usage, perplexity


class InvertibleLayerNorm(nn.Module):
    """LayerNorm that stores statistics for inverse transformation."""
    def __init__(self, num_features: int, eps: float = 1e-5):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))
        self.register_buffer('stored_mean', None, persistent=False)
        self.register_buffer('stored_std', None, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.stored_mean = x.mean(dim=(1, 2, 3), keepdim=True)
        self.stored_std = x.std(dim=(1, 2, 3), keepdim=True) + self.eps
        x_norm = (x - self.stored_mean) / self.stored_std
        return x_norm * self.weight + self.bias

    def inverse(self, x_norm: torch.Tensor) -> torch.Tensor:
        x = (x_norm - self.bias) / self.weight
        return x * self.stored_std + self.stored_mean


class RFSQStage(nn.Module):
    """Single stage of RFSQ with LayerNorm conditioning."""
    def __init__(self, levels: List[int]):
        super().__init__()
        self.fsq = FSQ(levels)
        self.layernorm = InvertibleLayerNorm(len(levels))

    def forward(self, residual: torch.Tensor):
        z_norm = self.layernorm(residual)
        z_q_norm, indices = self.fsq(z_norm)
        z_q = self.layernorm.inverse(z_q_norm)
        new_residual = residual - z_q
        return z_q, new_residual, indices


class RFSQ(nn.Module):
    """Residual FSQ with multiple stages."""
    def __init__(self, levels_per_stage: List[int], num_stages: int = 2):
        super().__init__()
        self.num_stages = num_stages
        self.stages = nn.ModuleList([RFSQStage(levels_per_stage) for _ in range(num_stages)])
        self._usage_indices = []

    def reset_usage(self):
        self._usage_indices = []

    def forward(self, z: torch.Tensor):
        residual = z
        z_q_sum = torch.zeros_like(z)
        all_indices = []
        for stage in self.stages:
            z_q, residual, indices = stage(residual)
            z_q_sum = z_q_sum + z_q
            all_indices.append(indices)
        self._usage_indices.append(all_indices)
        return z_q_sum, all_indices

    def get_usage_stats(self) -> Dict:
        if not self._usage_indices:
            return {}
        stats = {}
        for stage_idx in range(self.num_stages):
            all_stage_indices = torch.cat([b[stage_idx].flatten() for b in self._usage_indices])
            usage, perplexity = self.stages[stage_idx].fsq.get_codebook_usage(all_stage_indices)
            stats[f'stage{stage_idx}'] = (usage, perplexity)
        return stats

print("RFSQ modules defined")

## 4. VQ-VAE v14 Architecture

In [ ]:
class ResidualBlock3D(nn.Module):
    """3D residual block with BatchNorm."""
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv3d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(channels)
        self.conv2 = nn.Conv3d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + residual)


class EncoderV14(nn.Module):
    """Encoder: 32x32x32 -> 8x8x8 latent."""
    def __init__(self, in_channels: int = 40, hidden_dim: int = 128, rfsq_dim: int = 4):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv3d(in_channels, hidden_dim, 3, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.down1 = nn.Sequential(
            nn.Conv3d(hidden_dim, hidden_dim, 4, stride=2, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.res1 = ResidualBlock3D(hidden_dim)
        self.down2 = nn.Sequential(
            nn.Conv3d(hidden_dim, hidden_dim, 4, stride=2, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.res2 = nn.Sequential(*[ResidualBlock3D(hidden_dim) for _ in range(4)])
        self.proj = nn.Conv3d(hidden_dim, rfsq_dim, 3, padding=1)

    def forward(self, x):
        x = self.initial(x)
        x = self.res1(self.down1(x))
        x = self.res2(self.down2(x))
        return self.proj(x)


class DecoderV14(nn.Module):
    """Decoder: 8x8x8 latent -> 32x32x32 output with air logit bias."""
    def __init__(self, rfsq_dim: int = 4, hidden_dim: int = 128, num_blocks: int = 3717,
                 air_tokens: Set[int] = None, air_logit_bias: float = -2.0):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv3d(rfsq_dim, hidden_dim, 3, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.res1 = nn.Sequential(*[ResidualBlock3D(hidden_dim) for _ in range(4)])
        self.up1 = nn.Sequential(
            nn.ConvTranspose3d(hidden_dim, hidden_dim, 4, stride=2, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.res2 = ResidualBlock3D(hidden_dim)
        self.up2 = nn.Sequential(
            nn.ConvTranspose3d(hidden_dim, hidden_dim, 4, stride=2, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.final = nn.Conv3d(hidden_dim, num_blocks, 3, padding=1)
        
        if air_tokens and air_logit_bias != 0.0:
            with torch.no_grad():
                for token in air_tokens:
                    if token < num_blocks:
                        self.final.bias.data[token] = air_logit_bias
            print(f"  Air logit bias initialized to {air_logit_bias} for {len(air_tokens)} tokens")

    def forward(self, z_q):
        x = self.initial(z_q)
        x = self.res1(x)
        x = self.res2(self.up1(x))
        x = self.up2(x)
        return self.final(x)


class VQVAEv14(nn.Module):
    """VQ-VAE v14 with rebalanced loss weights."""
    def __init__(self, vocab_size: int = 3717, emb_dim: int = 40, hidden_dim: int = 128,
                 rfsq_levels: List[int] = None, num_stages: int = 2,
                 pretrained_embeddings: torch.Tensor = None,
                 air_tokens: Set[int] = None, air_logit_bias: float = -2.0):
        super().__init__()
        if rfsq_levels is None:
            rfsq_levels = [5, 5, 5, 5]
        self.rfsq_dim = len(rfsq_levels)
        self.vocab_size = vocab_size
        
        self.block_emb = nn.Embedding(vocab_size, emb_dim)
        if pretrained_embeddings is not None:
            self.block_emb.weight.data.copy_(pretrained_embeddings)
            self.block_emb.weight.requires_grad = False
        
        self.encoder = EncoderV14(emb_dim, hidden_dim, self.rfsq_dim)
        self.quantizer = RFSQ(rfsq_levels, num_stages)
        self.decoder = DecoderV14(
            self.rfsq_dim, hidden_dim, vocab_size,
            air_tokens=air_tokens, air_logit_bias=air_logit_bias
        )

    def forward(self, block_ids):
        z_e = self.encode(block_ids)
        z_q, indices = self.quantize(z_e)
        logits = self.decode(z_q)
        return logits, z_q, indices

    def encode(self, block_ids):
        x = self.block_emb(block_ids).permute(0, 4, 1, 2, 3)
        z_e = self.encoder(x).permute(0, 2, 3, 4, 1)
        return z_e

    def quantize(self, z_e):
        return self.quantizer(z_e)

    def decode(self, z_q):
        return self.decoder(z_q.permute(0, 4, 1, 2, 3))

print("VQ-VAE v14 architecture defined")

## 5. V14 Loss Function - Rebalanced Weights

**Key changes from v13b:**
- Structure->Air weight: 20.0 -> 6.0 (allow boundary adjustments)
- False air penalty: 10.0 -> 5.0 (was redundant)
- Hard volume weight: 10.0 -> 20.0 (make volume the priority)

**Trade-off**: Accept lower recall (~90%) for correct volume ratio.

In [ ]:
class V14Loss(nn.Module):
    """Rebalanced loss with lower asymmetric weight and stronger volume constraint.
    
    v14 key insight: The 20x asymmetric weight in v13b created a conflict.
    The model couldn't adjust air/structure boundaries to fix volume ratio.
    
    Solution: Reduce asymmetric weight (20x -> 6x) to allow boundary freedom,
    increase volume weight (10x -> 20x) to prioritize volume correctness.
    """
    def __init__(self, air_tokens: Set[int], gamma: float = 2.0,
                 structure_to_air_weight: float = 6.0,    # REDUCED from 20.0
                 air_to_structure_weight: float = 1.0,
                 false_air_penalty_weight: float = 5.0,   # REDUCED from 10.0
                 hard_volume_weight: float = 20.0):       # INCREASED from 10.0
        super().__init__()
        self.air_tokens = list(air_tokens)
        self.gamma = gamma
        self.structure_to_air_weight = structure_to_air_weight
        self.air_to_structure_weight = air_to_structure_weight
        self.false_air_penalty_weight = false_air_penalty_weight
        self.hard_volume_weight = hard_volume_weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor,
                z_q: torch.Tensor) -> Dict[str, torch.Tensor]:
        device = logits.device
        B, C, X, Y, Z = logits.shape
        
        air_tensor = torch.tensor(self.air_tokens, device=device, dtype=target.dtype)
        gt_is_air = torch.isin(target, air_tensor)
        gt_is_struct = ~gt_is_air
        
        logits_flat = logits.permute(0, 2, 3, 4, 1).reshape(-1, C)
        target_flat = target.reshape(-1)
        gt_is_struct_flat = gt_is_struct.reshape(-1)
        gt_is_air_flat = gt_is_air.reshape(-1)
        
        # Hard predictions (argmax)
        pred = logits_flat.argmax(dim=1)
        pred_is_air = torch.isin(pred, air_tensor)
        pred_is_struct = ~pred_is_air
        
        # === 1. Asymmetric Weighted Focal Cross-Entropy (REDUCED weights) ===
        weights = torch.ones(logits_flat.shape[0], device=device, dtype=torch.float)
        struct_to_air_mask = gt_is_struct_flat & pred_is_air
        weights[struct_to_air_mask] = self.structure_to_air_weight  # 6.0 now (was 20.0)
        air_to_struct_mask = gt_is_air_flat & pred_is_struct
        weights[air_to_struct_mask] = self.air_to_structure_weight
        
        log_probs = F.log_softmax(logits_flat, dim=1)
        probs = torch.exp(log_probs)
        p_t = probs.gather(1, target_flat.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - p_t) ** self.gamma
        
        ce_per_sample = F.cross_entropy(logits_flat, target_flat, reduction='none')
        weighted_focal_loss = (focal_weight * weights * ce_per_sample).mean()
        
        # === 2. False Air Penalty (REDUCED weight) ===
        if gt_is_struct_flat.any():
            struct_logits = logits_flat[gt_is_struct_flat]
            air_mask = torch.zeros(C, device=device, dtype=torch.bool)
            for t in self.air_tokens:
                if t < C:
                    air_mask[t] = True
            
            if air_mask.any():
                air_logits_at_struct = struct_logits[:, air_mask].max(dim=1)[0]
            else:
                air_logits_at_struct = torch.zeros(struct_logits.shape[0], device=device)
            
            non_air_logits = struct_logits.clone()
            non_air_logits[:, air_mask] = float('-inf')
            max_non_air = non_air_logits.max(dim=1)[0]
            
            margin = 1.0
            violation = F.relu(air_logits_at_struct - max_non_air + margin)
            false_air_penalty = violation.mean()
        else:
            false_air_penalty = torch.tensor(0.0, device=device)
        
        # === 3. HARD Volume Penalty using STE (INCREASED weight) ===
        # Straight-Through Estimator: forward uses hard, backward uses soft
        hard_one_hot = F.one_hot(pred, num_classes=C).float()
        soft_probs = F.softmax(logits_flat, dim=1)
        ste_probs = soft_probs + (hard_one_hot - soft_probs).detach()
        
        air_mask_full = torch.zeros(C, device=device, dtype=torch.bool)
        for t in self.air_tokens:
            if t < C:
                air_mask_full[t] = True
        
        struct_probs = ste_probs.clone()
        struct_probs[:, air_mask_full] = 0
        hard_pred_volume = struct_probs.sum(dim=1).mean()
        gt_volume = gt_is_struct_flat.float().mean()
        hard_volume_loss = torch.abs(hard_pred_volume - gt_volume)
        
        # === Total Loss ===
        total_loss = (
            weighted_focal_loss +
            self.false_air_penalty_weight * false_air_penalty +
            self.hard_volume_weight * hard_volume_loss
        )
        
        # === Metrics ===
        with torch.no_grad():
            correct = (pred == target_flat)
            accuracy = correct.float().mean()
            building_acc = correct[gt_is_struct_flat].float().mean() if gt_is_struct_flat.any() else torch.tensor(0.0)
            air_acc = correct[gt_is_air_flat].float().mean() if gt_is_air_flat.any() else torch.tensor(0.0)
            recall = pred_is_struct[gt_is_struct_flat].float().mean() if gt_is_struct_flat.any() else torch.tensor(0.0)
            false_air_rate = pred_is_air[gt_is_struct_flat].float().mean() if gt_is_struct_flat.any() else torch.tensor(0.0)
            pred_volume = pred_is_struct.float().sum()
            gt_volume_hard = gt_is_struct_flat.float().sum()
            volume_ratio = pred_volume / (gt_volume_hard + 1e-6)
            precision = correct[pred_is_struct & gt_is_struct_flat].sum().float() / (pred_is_struct.sum().float() + 1e-6)
        
        return {
            'loss': total_loss,
            'focal_loss': weighted_focal_loss.detach(),
            'false_air_penalty': false_air_penalty.detach(),
            'hard_volume_loss': hard_volume_loss.detach(),
            'hard_pred_volume': hard_pred_volume.detach(),
            'gt_volume': gt_volume.detach(),
            'volume_ratio': volume_ratio,
            'accuracy': accuracy,
            'building_acc': building_acc,
            'air_acc': air_acc,
            'recall': recall,
            'precision': precision,
            'false_air_rate': false_air_rate,
        }

print("V14Loss defined (Rebalanced Weights)")
print(f"  - Structure->Air weight: 6.0x (was 20.0x in v13b)")
print(f"  - False air penalty: 5.0x (was 10.0x in v13b)")
print(f"  - Hard volume penalty: 20.0x (was 10.0x in v13b)")
print(f"  - Method: STE (same as v13b)")

## 6. Configuration

In [ ]:
# === Data Paths (using LOCAL storage for speed) ===
DATA_DIR = "/content/splits/train"
VAL_DIR = "/content/splits/val"
VOCAB_PATH = "/content/vocabulary/tok2block.json"
V3_EMBEDDINGS_PATH = "/content/embeddings/block_embeddings_v3.npy"

print("Checking paths...")
for name, path in [('DATA_DIR', DATA_DIR), ('VAL_DIR', VAL_DIR),
                   ('VOCAB_PATH', VOCAB_PATH), ('V3_EMBEDDINGS_PATH', V3_EMBEDDINGS_PATH)]:
    exists = Path(path).exists()
    print(f"  {name}: {'[OK]' if exists else '[NOT FOUND]'}")

# === v14 Architecture (unchanged) ===
HIDDEN_DIM = 128
RFSQ_LEVELS = [5, 5, 5, 5]
NUM_STAGES = 2
AIR_LOGIT_BIAS = -2.0

# === v14 Loss Weights (REBALANCED) ===
FOCAL_GAMMA = 2.0
STRUCTURE_TO_AIR_WEIGHT = 6.0    # REDUCED from 20.0
AIR_TO_STRUCTURE_WEIGHT = 1.0
FALSE_AIR_PENALTY_WEIGHT = 5.0   # REDUCED from 10.0
HARD_VOLUME_WEIGHT = 20.0        # INCREASED from 10.0

# === Training ===
EPOCHS = 25
BATCH_SIZE = 4
BASE_LR = 2e-4
USE_AMP = True
GRAD_ACCUM_STEPS = 4
SEED = 42
NUM_WORKERS = 2  # Can use workers with local storage

print(f"\n{'='*60}")
print("V14 CONFIGURATION - REBALANCED LOSS WEIGHTS")
print(f"{'='*60}")
print(f"Goal: Fix volume ratio by allowing boundary adjustments")
print(f"\nKey changes from v13b:")
print(f"  - Structure->Air: 6.0x (was 20.0x) - allow boundaries to adjust")
print(f"  - False air penalty: 5.0x (was 10.0x) - was redundant")
print(f"  - Hard volume: 20.0x (was 10.0x) - volume is priority")
print(f"\nExpected trade-off:")
print(f"  - Recall: 99.9% -> ~90% (acceptable)")
print(f"  - Volume ratio: 1.23x -> 0.9-1.1x (goal!)")
print(f"\nTarget: volume_ratio 0.9-1.1x, building_acc >= 45%")
print(f"{'='*60}")

## 7. Load Vocabulary and Embeddings

In [ ]:
with open(VOCAB_PATH, 'r') as f:
    tok2block = {int(k): v for k, v in json.load(f).items()}

VOCAB_SIZE = len(tok2block)
print(f"Vocabulary size: {VOCAB_SIZE}")

AIR_TOKENS: Set[int] = set()
for tok, block in tok2block.items():
    if 'air' in block.lower() and 'stair' not in block.lower():
        AIR_TOKENS.add(tok)
        print(f"  Air token: {tok} = {block}")

AIR_TOKENS_TENSOR = torch.tensor(sorted(AIR_TOKENS), dtype=torch.long)

v3_embeddings = np.load(V3_EMBEDDINGS_PATH).astype(np.float32)
EMBEDDING_DIM = v3_embeddings.shape[1]
print(f"\nEmbeddings: {v3_embeddings.shape}")

## 8. Dataset

In [ ]:
class VQVAEDataset(Dataset):
    def __init__(self, data_dir: str):
        self.h5_files = sorted(Path(data_dir).glob("*.h5"))
        if not self.h5_files:
            raise ValueError(f"No H5 files in {data_dir}")
        print(f"Found {len(self.h5_files)} structures")

    def __len__(self):
        return len(self.h5_files)

    def __getitem__(self, idx):
        with h5py.File(self.h5_files[idx], 'r') as f:
            structure = f[list(f.keys())[0]][:].astype(np.int64)
        return torch.from_numpy(structure).long()

train_dataset = VQVAEDataset(DATA_DIR)
val_dataset = VQVAEDataset(VAL_DIR)

## 9. Create Model, Criterion, Optimizer

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

model = VQVAEv14(
    vocab_size=VOCAB_SIZE,
    emb_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    rfsq_levels=RFSQ_LEVELS,
    num_stages=NUM_STAGES,
    pretrained_embeddings=torch.from_numpy(v3_embeddings),
    air_tokens=AIR_TOKENS,
    air_logit_bias=AIR_LOGIT_BIAS,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

criterion = V14Loss(
    air_tokens=AIR_TOKENS,
    gamma=FOCAL_GAMMA,
    structure_to_air_weight=STRUCTURE_TO_AIR_WEIGHT,
    air_to_structure_weight=AIR_TO_STRUCTURE_WEIGHT,
    false_air_penalty_weight=FALSE_AIR_PENALTY_WEIGHT,
    hard_volume_weight=HARD_VOLUME_WEIGHT,
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

print("\nModel, criterion, optimizer created")

## 10. Data Loaders

In [ ]:
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, generator=g)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

## 11. Training Functions

In [ ]:
def train_epoch(model, criterion, loader, optimizer, scaler, device):
    model.train()
    model.quantizer.reset_usage()
    
    metrics_sum = defaultdict(float)
    n = 0
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(tqdm(loader, desc="Train", leave=False)):
        batch = batch.to(device)
        
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits, z_q, indices = model(batch)
            loss_dict = criterion(logits, batch, z_q)
            loss = loss_dict['loss'] / GRAD_ACCUM_STEPS
        
        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        with torch.no_grad():
            for k, v in loss_dict.items():
                metrics_sum[k] += v.item() if torch.is_tensor(v) else v
        n += 1
    
    metrics = {k: v / n for k, v in metrics_sum.items()}
    for name, (usage, perp) in model.quantizer.get_usage_stats().items():
        metrics[f'{name}_usage'] = usage
        metrics[f'{name}_perplexity'] = perp
    return metrics


@torch.no_grad()
def validate(model, criterion, loader, device):
    model.eval()
    model.quantizer.reset_usage()
    
    metrics_sum = defaultdict(float)
    n = 0
    
    for batch in tqdm(loader, desc="Val", leave=False):
        batch = batch.to(device)
        
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits, z_q, indices = model(batch)
            loss_dict = criterion(logits, batch, z_q)
        
        for k, v in loss_dict.items():
            metrics_sum[k] += v.item() if torch.is_tensor(v) else v
        n += 1
    
    metrics = {k: v / n for k, v in metrics_sum.items()}
    for name, (usage, perp) in model.quantizer.get_usage_stats().items():
        metrics[f'{name}_usage'] = usage
        metrics[f'{name}_perplexity'] = perp
    return metrics

print("Training functions defined")

## 12. Training Loop

In [ ]:
print("="*70)
print("VQ-VAE V14 TRAINING - REBALANCED LOSS WEIGHTS")
print("="*70)
print(f"Goal: Fix volume ratio by allowing boundary adjustments")
print(f"Key changes: Struct->Air 6x (was 20x), Hard volume 20x (was 10x)")
print(f"Epochs: {EPOCHS}")
print("="*70)

history = {
    'train_loss': [], 'val_loss': [],
    'train_focal_loss': [], 'val_focal_loss': [],
    'train_false_air_penalty': [], 'val_false_air_penalty': [],
    'train_hard_volume_loss': [], 'val_hard_volume_loss': [],
    'train_hard_pred_volume': [], 'val_hard_pred_volume': [],
    'train_gt_volume': [], 'val_gt_volume': [],
    'train_volume_ratio': [], 'val_volume_ratio': [],
    'train_recall': [], 'val_recall': [],
    'train_precision': [], 'val_precision': [],
    'train_accuracy': [], 'val_accuracy': [],
    'train_building_acc': [], 'val_building_acc': [],
    'train_air_acc': [], 'val_air_acc': [],
    'train_false_air_rate': [], 'val_false_air_rate': [],
    'train_stage0_perplexity': [], 'val_stage0_perplexity': [],
    'learning_rate': [],
}

best_metric = 0
best_epoch = 0
start_time = time.time()

for epoch in range(EPOCHS):
    train_m = train_epoch(model, criterion, train_loader, optimizer, scaler, device)
    val_m = validate(model, criterion, val_loader, device)
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    for prefix, m in [('train', train_m), ('val', val_m)]:
        history[f'{prefix}_loss'].append(m.get('loss', 0))
        history[f'{prefix}_focal_loss'].append(m.get('focal_loss', 0))
        history[f'{prefix}_false_air_penalty'].append(m.get('false_air_penalty', 0))
        history[f'{prefix}_hard_volume_loss'].append(m.get('hard_volume_loss', 0))
        history[f'{prefix}_hard_pred_volume'].append(m.get('hard_pred_volume', 0))
        history[f'{prefix}_gt_volume'].append(m.get('gt_volume', 0))
        history[f'{prefix}_volume_ratio'].append(m.get('volume_ratio', 0))
        history[f'{prefix}_recall'].append(m.get('recall', 0))
        history[f'{prefix}_precision'].append(m.get('precision', 0))
        history[f'{prefix}_accuracy'].append(m.get('accuracy', 0))
        history[f'{prefix}_building_acc'].append(m.get('building_acc', 0))
        history[f'{prefix}_air_acc'].append(m.get('air_acc', 0))
        history[f'{prefix}_false_air_rate'].append(m.get('false_air_rate', 0))
        history[f'{prefix}_stage0_perplexity'].append(m.get('stage0_perplexity', 0))
    history['learning_rate'].append(current_lr)
    
    # Best model: balance volume ratio and accuracy
    vol_score = 1.0 - abs(val_m['volume_ratio'] - 1.0)
    acc_score = val_m.get('building_acc', val_m.get('accuracy', 0))
    current_metric = 0.5 * vol_score + 0.5 * acc_score
    
    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch + 1
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/vqvae_v14_best.pt")
    
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/vqvae_v14_epoch{epoch+1}.pt")
    
    vol_str = f"{val_m['volume_ratio']:.3f}x"
    vol_ok = 0.9 <= val_m['volume_ratio'] <= 1.1
    
    print(f"E{epoch+1:2d} | Build: {val_m['building_acc']:.1%} | "
          f"Vol: {vol_str} {'OK' if vol_ok else '!!'} | "
          f"Recall: {val_m['recall']:.1%} | FAR: {val_m['false_air_rate']:.1%}")

train_time = time.time() - start_time
print(f"\n{'='*70}")
print("TRAINING COMPLETE")
print(f"{'='*70}")
print(f"Time: {train_time/60:.1f} minutes")
print(f"Best epoch: {best_epoch}")
print(f"Final volume ratio: {history['val_volume_ratio'][-1]:.3f}x")
print(f"Final recall: {history['val_recall'][-1]:.1%}")
print(f"Final false air rate: {history['val_false_air_rate'][-1]:.1%}")
print(f"Final building accuracy: {history['val_building_acc'][-1]:.1%}")

## 13. Plot Training Results

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
epochs_range = range(1, len(history['train_loss']) + 1)

# 1. Volume Ratio (KEY METRIC)
ax = axes[0, 0]
ax.plot(epochs_range, history['train_volume_ratio'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_volume_ratio'], 'r--', label='Val', linewidth=2)
ax.axhline(y=1.0, color='g', linestyle='--', linewidth=2, label='Target (1.0x)')
ax.axhline(y=1.307, color='orange', linestyle=':', alpha=0.7, label='v11 (1.307x)')
ax.axhline(y=1.233, color='purple', linestyle=':', alpha=0.7, label='v13b (1.233x)')
ax.fill_between(epochs_range, 0.9, 1.1, alpha=0.2, color='green', label='Target zone')
ax.set_xlabel('Epoch'); ax.set_ylabel('Volume Ratio')
ax.set_title('Volume Ratio (KEY METRIC)', fontweight='bold', fontsize=12, color='darkred')
ax.legend(loc='upper right', fontsize=8); ax.grid(True, alpha=0.3)
ax.set_ylim(0, max(2.0, max(history['val_volume_ratio']) * 1.1))

# 2. Hard Volume Prediction vs GT
ax = axes[0, 1]
ax.plot(epochs_range, history['train_hard_pred_volume'], 'b-', label='Train Pred', linewidth=2)
ax.plot(epochs_range, history['val_hard_pred_volume'], 'r--', label='Val Pred', linewidth=2)
ax.plot(epochs_range, history['val_gt_volume'], 'g:', label='Val GT', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Normalized Volume')
ax.set_title('Hard Volume Prediction vs GT (STE)', fontweight='bold', fontsize=12)
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)

# 3. False Air Rate
ax = axes[0, 2]
ax.plot(epochs_range, history['train_false_air_rate'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_false_air_rate'], 'r--', label='Val', linewidth=2)
ax.axhline(y=0.10, color='g', linestyle='--', alpha=0.5, label='Target (<10%)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Rate')
ax.set_title('False Air Rate', fontweight='bold', fontsize=12)
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)

# 4. Recall
ax = axes[1, 0]
ax.plot(epochs_range, history['train_recall'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_recall'], 'r--', label='Val', linewidth=2)
ax.axhline(y=0.85, color='g', linestyle='--', alpha=0.5, label='Target (85%)')
ax.axhline(y=0.999, color='orange', linestyle=':', alpha=0.5, label='v13b (99.8%)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Recall')
ax.set_title('Recall (expect lower than v13b)', fontweight='bold', fontsize=12)
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)

# 5. Building Accuracy
ax = axes[1, 1]
ax.plot(epochs_range, history['train_building_acc'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_building_acc'], 'r--', label='Val', linewidth=2)
ax.axhline(y=0.556, color='orange', linestyle=':', alpha=0.5, label='v11 (55.6%)')
ax.axhline(y=0.073, color='purple', linestyle=':', alpha=0.5, label='v13b (7.3%)')
ax.axhline(y=0.45, color='g', linestyle='--', alpha=0.5, label='Target (45%)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Building Accuracy', fontweight='bold', fontsize=12)
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)

# 6. Precision
ax = axes[1, 2]
ax.plot(epochs_range, history['train_precision'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_precision'], 'r--', label='Val', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Precision')
ax.set_title('Precision', fontweight='bold', fontsize=12)
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)

# 7. Loss Components
ax = axes[2, 0]
ax.plot(epochs_range, history['val_focal_loss'], 'b-', label='Focal CE', linewidth=2)
ax.plot(epochs_range, [x * FALSE_AIR_PENALTY_WEIGHT for x in history['val_false_air_penalty']], 'r-', label=f'False Air ({FALSE_AIR_PENALTY_WEIGHT}x)', linewidth=2)
ax.plot(epochs_range, [x * HARD_VOLUME_WEIGHT for x in history['val_hard_volume_loss']], 'g-', label=f'Hard Vol ({HARD_VOLUME_WEIGHT}x)', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Loss Components (Val, weighted)', fontweight='bold', fontsize=12)
ax.legend(); ax.grid(True, alpha=0.3)

# 8. Total Loss
ax = axes[2, 1]
ax.plot(epochs_range, history['train_loss'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_loss'], 'r--', label='Val', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Total Loss', fontweight='bold', fontsize=12)
ax.legend(); ax.grid(True, alpha=0.3)

# 9. RFSQ Perplexity
ax = axes[2, 2]
ax.plot(epochs_range, history['train_stage0_perplexity'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_stage0_perplexity'], 'r--', label='Val', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Perplexity')
ax.set_title('RFSQ Perplexity', fontweight='bold', fontsize=12)
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/vqvae_v14_training.png", dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("\n" + "="*70)
print("V14 RESULTS SUMMARY")
print("="*70)
final_vol = history['val_volume_ratio'][-1]
final_recall = history['val_recall'][-1]
final_build = history['val_building_acc'][-1]
final_far = history['val_false_air_rate'][-1]

print(f"Final volume ratio: {final_vol:.3f}x (v11: 1.307x, v13b: 1.233x)")
print(f"Final recall: {final_recall:.1%} (v13b: 99.8%)")
print(f"Final false air rate: {final_far:.1%}")
print(f"Final building accuracy: {final_build:.1%} (v11: 55.6%, v13b: 7.3%)")
print()

vol_ok = 0.9 <= final_vol <= 1.1
recall_ok = final_recall >= 0.85
far_ok = final_far < 0.10
acc_ok = final_build >= 0.45

print("Success Criteria:")
print(f"  Volume 0.9-1.1x: {'PASS' if vol_ok else 'FAIL'} ({final_vol:.3f}x)")
print(f"  Recall >= 85%: {'PASS' if recall_ok else 'FAIL'} ({final_recall:.1%})")
print(f"  FAR < 10%: {'PASS' if far_ok else 'FAIL'} ({final_far:.1%})")
print(f"  Build acc >= 45%: {'PASS' if acc_ok else 'FAIL'} ({final_build:.1%})")
print()

if vol_ok and recall_ok and far_ok and acc_ok:
    print("SUCCESS! All v14 targets met.")
elif vol_ok:
    print("GOOD: Volume ratio fixed! Check other metrics.")
elif final_vol < 0.9:
    print("UNDER-PREDICTING: Try asymmetric weight = 8x")
elif final_vol > 1.1:
    print("OVER-PREDICTING: Try hard volume weight = 30x")
else:
    print("Mixed results - analyze loss curves.")

## 14. Save Results

In [ ]:
results = {
    'config': {
        'version': 'v14-REBALANCED-WEIGHTS',
        'latent_resolution': '8x8x8',
        'hidden_dim': HIDDEN_DIM,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'base_lr': BASE_LR,
        'focal_gamma': FOCAL_GAMMA,
        'structure_to_air_weight': STRUCTURE_TO_AIR_WEIGHT,
        'air_to_structure_weight': AIR_TO_STRUCTURE_WEIGHT,
        'false_air_penalty_weight': FALSE_AIR_PENALTY_WEIGHT,
        'hard_volume_weight': HARD_VOLUME_WEIGHT,
        'method': 'straight-through-estimator',
        'air_logit_bias': AIR_LOGIT_BIAS,
        'seed': SEED,
    },
    'results': {
        'final_volume_ratio': float(history['val_volume_ratio'][-1]),
        'final_recall': float(history['val_recall'][-1]),
        'final_precision': float(history['val_precision'][-1]),
        'final_building_acc': float(history['val_building_acc'][-1]),
        'final_air_acc': float(history['val_air_acc'][-1]),
        'final_false_air_rate': float(history['val_false_air_rate'][-1]),
        'final_hard_pred_volume': float(history['val_hard_pred_volume'][-1]),
        'final_gt_volume': float(history['val_gt_volume'][-1]),
        'best_epoch': best_epoch,
        'training_time_min': float(train_time / 60),
        'volume_target_met': bool(0.9 <= history['val_volume_ratio'][-1] <= 1.1),
        'recall_target_met': bool(history['val_recall'][-1] >= 0.85),
        'far_target_met': bool(history['val_false_air_rate'][-1] < 0.10),
        'accuracy_target_met': bool(history['val_building_acc'][-1] >= 0.45),
    },
    'history': {k: [float(x) for x in v] for k, v in history.items()},
}

with open(f"{OUTPUT_DIR}/vqvae_v14_results.json", 'w') as f:
    json.dump(results, f, indent=2)

torch.save(model.state_dict(), f"{OUTPUT_DIR}/vqvae_v14_final.pt")

print("Results saved:")
print(f"  {OUTPUT_DIR}/vqvae_v14_results.json")
print(f"  {OUTPUT_DIR}/vqvae_v14_best.pt")
print(f"  {OUTPUT_DIR}/vqvae_v14_final.pt")
print(f"  {OUTPUT_DIR}/vqvae_v14_training.png")
print("\nDone!")